## PILLAR 6: NETWORKING — VOCABULARY

### Key Networking Definitions

The diagnosis paths in `ip addr` / `ip route` only make sense once these terms are solid — "same subnet" vs "needs a gateway" is the single most common branch point in any connectivity ticket. This notebook is a running reference — add new terms here as they come up across this pillar (and others).

---

### LAN (Local Area Network)
[Wiki](https://en.wikipedia.org/wiki/Local_area_network)

A group of devices that can talk to each other **directly**, without going through a router — same physical/virtual network segment, same subnet. On your WSL box, "LAN" in the traditional sense is actually your *Windows host's* real network (Wi-Fi/Ethernet); WSL2's `eth0` sits on its own private virtual LAN behind NAT (more below).

**Developer bridge:** think of a LAN like containers on the same Docker bridge network — they can reach each other by IP directly, no external routing needed, because Docker's bridge is functionally a mini-LAN.

---

### Subnet / CIDR notation

A subnet is a mathematically-defined slice of IP address space — "these addresses are considered local to each other." Written as `IP/prefix`, e.g. `172.26.112.0/20`. The `/20` says: the first 20 bits are the **network portion** (fixed, shared by everyone on this subnet), the remaining 12 bits are the **host portion** (unique per device). A `/20` gives you 4096 addresses in that range.

**WHAT TO LOOK FOR:** two IPs are on the same subnet if, after applying the mask, their network portion matches. `172.26.126.1/20` and `172.26.112.1/20` share the same first 20 bits → same subnet → no gateway needed to talk to each other. This is exactly what `scope link` in `ip route` output tells you.

---

### Gateway / Default Gateway

A device (usually a router) that sits on your subnet *and also* has a connection to another network (often "the rest of the internet"). It's the "exit door." When a packet's destination isn't on your local subnet, the kernel hands it to the gateway and says "you figure out the next step." The **default** gateway is the one used when no more specific route matches — the catch-all.

---

### WAN (Wide Area Network)

Everything past your gateway — the broader internet, or a company's network spanning multiple sites. You never talk to a WAN destination directly; you always go through your gateway first.

---

### NAT (Network Address Translation)

A technique where a router/gateway rewrites packets so a whole private network (with private, non-internet-routable addresses like `172.x`, `192.168.x`, `10.x`) can share one public IP to reach the outside world. This is *exactly* what WSL2 does to you: your `eth0` address (`172.26.126.1`) is private, valid only inside the WSL virtual network. When you reach the real internet, WSL's virtual router NATs your traffic behind the Windows host's actual network connection.

**Developer bridge:** NAT is like a reverse proxy that rewrites the source address on the way out and remembers how to route the response back — conceptually similar to how a load balancer rewrites/tracks connections for backend pods that don't have public IPs themselves.

---

### Private vs Public IP ranges

Certain ranges are reserved and never routable on the public internet — used for internal/LAN addressing everywhere, which is why they can safely repeat across millions of networks without collision:

|Range|CIDR|
|---|---|
|`10.0.0.0` – `10.255.255.255`|`10.0.0.0/8`|
|`172.16.0.0` – `172.31.255.255`|`172.16.0.0/12`|
|`192.168.0.0` – `192.168.255.255`|`192.168.0.0/16`|

**WHAT TO LOOK FOR:** a `172.26.x.x` address falls inside `172.16.0.0/12` — confirms it's a private address, consistent with it being WSL2's internal NAT network, not a real internet-facing IP.

---

### Interface scope, recap

`scope link` (seen in `ip route` output) = "reachable directly on this link/subnet, no gateway hop." `scope global` (seen on the `inet` line in `ip addr`) = "this address is usable for general routing, not just loopback/link-local." Different meaning depending on whether it's labeling a route or an address, but same root idea: how far can this thing be reached without help.

---

**TICKET SCENARIO tying it together:** "Two servers on the same rack can't reach each other, but both can reach the internet fine." Check both machines' IP/subnet with `ip addr` — if their addresses don't actually fall in the same subnet (e.g., one's a `/24` and the other's a `/25` and they land on different sides of that boundary), they'll try to route to each other *through the gateway* instead of directly — and if the gateway/firewall doesn't permit that hairpin traffic, they can't talk despite being on the "same" physical switch. This is a classic case where the LAN/subnet definition isn't just theory — it changes the actual path packets take.

**GOTCHA:** "same LAN" and "same subnet" get used interchangeably but aren't strictly identical — you can have multiple subnets on one physical LAN (VLANs), where devices are physically on the same wire but the OS still routes between them via a gateway because their subnet definitions differ.

**TRY NOW:**

In [ ]:
dennie@X390:~/linux-tut$ ip addr show eth0 | grep inet
    inet 172.30.240.133/20 brd 172.30.255.255 scope global eth0
    inet6 fe80::215:5dff:feaf:e924/64 scope link

In [ ]:
dennie@X390:~/linux-tut$ ip route | grep default
default via 172.30.240.1 dev eth0 proto kernel 

Confirm for yourself: does your `eth0`'s subnet (from the first command) match the subnet shown in the non-default route line from `ip route`? It should — that route was auto-derived from your interface config.